In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

from datetime import datetime
import matplotlib.pyplot as plt

import base64

from datetime import datetime, UTC


import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# 
# LOAD ENPARTO LOGO
# 
image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

- from the perspective of a GEN MP: 
    - how much is produced
    - how much is distributed in the REC
    - how much is now distributed in the REC
    - how much is now distributed in X
    - how much is distributed IN TOTAL  / change of total surplus

- from the perspective of a CONS MP: 
    - how much is consumed
    - how much is covered by the REC
    - how much is now covered by the REC
    - how much is now covered by the REC
    - how much is covered IN TOTAL  / change of total Comm Cov

- from the perspective of a single REC:
    - sums of all energy flow values
    - sums of all energy flow INTERNAL values
    - sums of all energy flow to X values

## Params

In [ ]:
# # PARMS
# changeable
org_ids = [
    1,
    2,
    13,
    17,
    4,
    8,
    9,
    10,
    11,
    12,
    14,
    15,
    16,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    26,
    27,
    29,
    31,
    32,
    33,
    34,
    35,
    36,
    63,
    64,
    65,
    163,
    166,
    167,
    168,
    169,
    172,
]

start_time = datetime(2025, 1, 2)
end_time = datetime(2025, 8, 30)

# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

## load data

In [ ]:
config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename_template = config_dict["SINGLE_MPS_OF_EEG"]

In [ ]:
USECOLS = [
    "time",
    "organization_id",
    "metering_point_id",
    "wt_meas_cons",
    "wt_meas_gen",
    "wt_surp_gen",
]

DTYPES = {
    "organization_id": "int32",
    "metering_point_id": "int32",
    "wt_meas_cons": "float32",
    "wt_meas_gen": "float32",
    "wt_surp_gen": "float32",
}


In [ ]:
raw_eeg = []

for act_org_id in org_ids:
    act_filepath_to_load = (
        f"{path_to_local_data}{input_filename_template.format(org_id=act_org_id)}"
    )
    act_df = pd.read_csv(act_filepath_to_load)
    raw_eeg.append(act_df)

# row-wise append into a single DataFrame
raw_eeg_df = pd.concat(raw_eeg, ignore_index=True)

raw_eeg_df['time'] = pd.to_datetime(raw_eeg_df['time'], utc=True)

eeg_selected_time_horizon = raw_eeg_df[
    (raw_eeg_df["time"] > pd.Timestamp(start_time, tz='UTC')) &
    (raw_eeg_df["time"] < pd.Timestamp(end_time, tz='UTC'))
]
del raw_eeg_df



print(f"{eeg_selected_time_horizon.dtypes}")
print(f"len: {len(eeg_selected_time_horizon)}")

In [ ]:
eeg_selected_feat = eeg_selected_time_horizon[["time", "organization_id", "metering_point_id", "wt_meas_cons", "wt_meas_gen", "wt_surp_gen"]].copy()
del eeg_selected_time_horizon

# PRINT DF INFO
# 
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.drop(["organization_id", "metering_point_id"], axis=1).describe())

In [ ]:
work = eeg_selected_feat.drop_duplicates().copy()
del eeg_selected_feat

In [ ]:
eeg_agg = work.groupby(["time", "organization_id"]).sum(numeric_only=True).reset_index()
eeg_agg["is_surp"] = eeg_agg["wt_surp_gen"] > 0
eeg_agg["is_cons_or_gen"] = (
    (eeg_agg["wt_meas_cons"] > 0) | (eeg_agg["wt_meas_gen"] > 0)
)

temp_eeg_bool_agg = (
    eeg_agg
    .groupby([eeg_agg["time"].dt.date, "organization_id"])
    .agg(
        all_day_surplus=("is_surp", "all"),
        any_cons_or_gen=("is_cons_or_gen", "any"),
        no_surplus=("is_surp", lambda x: (~x).all())
    )
    .reset_index()
)

temp_eeg_bool_agg["all_day_deficit"] = (
    temp_eeg_bool_agg["no_surplus"] &
    temp_eeg_bool_agg["any_cons_or_gen"]
)

temp_eeg_bool_agg.drop(columns=["no_surplus", "any_cons_or_gen"], inplace=True)


In [ ]:
df

In [ ]:
org_ids

In [ ]:


# example data for multiple org IDs
df = temp_eeg_bool_agg.rename(columns={"organization_id":"org_id", "time":"date"})

# colour logic
def get_color(row):
    if row["all_day_surplus"]:
        return "green"
    elif row["all_day_deficit"]:
        return "red"
    else:
        return "white"

df["color"] = df.apply(get_color, axis=1)

# prepare bar width and position
org_ids = (
    df.groupby("org_id")["date"]
      .min()                      # earliest date per org
      .reset_index()
      .sort_values(["date", "org_id"], ascending=[False, False])  # first by start date, then org_id
      ["org_id"]
      .tolist()
)
date_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
bar_height = 0.8

fig, ax = plt.subplots(figsize=(15, 8))

# draw one rectangle per org / day
for i, org in enumerate(org_ids):
    org_data = df[df["org_id"] == org]
    for _, row in org_data.iterrows():
        ax.barh(
            y=i, 
            width=1,  # 1 day wide
            left=row["date"], 
            height=bar_height, 
            color=row["color"],
            edgecolor="black"  # optional: border
        )

# adjust axes
ax.set_yticks(range(len(org_ids)))
ax.set_yticklabels(org_ids)
ax.set_xlim(date_range.min(), date_range.max() + pd.Timedelta(days=1))
ax.set_xlabel("Date")
ax.set_ylabel("Org ID")
ax.set_title("All-Day Surplus (green) / Deficit (red) per Org and Day")

# legend
handles = [
    mpatches.Patch(color='green', label='Surplus'),
    mpatches.Patch(color='red', label='Deficit'),
    mpatches.Patch(color='white', label='Neutral'),
]
ax.legend(
    handles=handles,
    loc='upper left',   # top left
    bbox_to_anchor=(0, 1),  # slightly above the plot, if needed
    ncol=len(handles),  # side by side
    frameon=False       # optional: no frame
)

plt.show()
